# Lev-3B — log anomaly decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Lev-3B-log-anomaly-decisions.ipynb)

Run Cisco's [`cisco-ai/lev-3b`](https://huggingface.co/cisco-ai/lev-3b) with the **Lev runtime** (`kev.evaluate.load` + `kev.api.SystemOneRequest`): give it a shared **state** (a window of log lines) and named questions, and get a **probability vector over the alternatives you listed**. It does **not** generate answer text.

Lev-3B is a compact Apache-2.0 **probabilistic decision model**: LoRA (`r=16`, `alpha=32`) plus a trained **pointer head** (`head.pt`) on [`ibm-granite/granite-4.1-3b-base`](https://huggingface.co/ibm-granite/granite-4.1-3b-base), pinned to revision `dacb9cb9157bec98e99b09f285c92a4d58405c96`. The Hub files are the adapter, the trained tokenizer (with delimiter tokens), `head.pt`, and optional `temperature.json`. Base weights are **not** in the repo; the Lev loader fetches that Granite revision automatically.

Question types from the [card](https://huggingface.co/cisco-ai/lev-3b):

| Type | Alternatives | Probability-vector order |
| --- | --- | --- |
| `noul` | Binary, with optional descriptions | `false`, then `true` |
| `choice` | 1–255 named alternatives in `criteria` | Dictionary insertion order |
| `score` | 2–255 ordered descriptions in `criteria` | List order |

**Why this model.** The card adapted this checkpoint for log analysis (anomaly detection, ATT&CK parent-technique classification, and general-log questions). A trained pointer head scores the described alternatives instead of sampling tokens. Ordinary `AutoModelForCausalLM` or PEFT-only loading does **not** restore the pointer readout — use Lev.

See [use cases](https://github.com/donvito/notebooks/blob/main/docs/Lev-3B-log-anomaly-decisions.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0 for the Lev adapter and pointer head. The Granite base remains under its own terms (also Apache-2.0). Public weights — cancel any Colab `HF_TOKEN` prompt.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** VRAM, compute capability **7.5**: **no native bfloat16**, **no FlashAttention 2**. The card loads `dtype=torch.bfloat16`. This notebook loads the backbone in **float16**. Adapters, structural-token rows, and the pointer head stay **float32**, as the card specifies. Download is about **7 GB** (Granite base ~6.8 GB + Lev files ~0.14 GB). Expected peak VRAM on these short demos is about **8–10 GB** — an estimate, not a measured Colab run. Float32 for the whole 3B backbone is ~12 GB and is a last resort if float16 misbehaves.

**Install (Colab, not uv).** The card clones [`splunk/lev`](https://github.com/splunk/lev), runs `python3 scripts/setup_kev.py`, and uses a uv env with a pinned torch extra. Colab already has Python / torch / CUDA, so this notebook installs the patched `kev` package **into Colab's environment** (`pip install -e`) and **keeps Colab's torch**. Do **not** install `flash-attn`, `flash-linear-attention`, or `causal-conv1d`.

**Links:** [model card](https://huggingface.co/cisco-ai/lev-3b) · [paper](https://arxiv.org/abs/2610.38915) · [splunk/lev](https://github.com/splunk/lev) · [Kev](https://github.com/jaredpalmer/kev) · [announcement](https://x.com/LiangGou/status/2124705890242838645) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Lev-3B-log-anomaly-decisions.md)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so the later transformers pin can take effect. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Card default is bfloat16. T4 is Turing sm_75 — stay on float16.
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager)")
if major < 8:
    print("No native bfloat16 on this GPU; loading the Granite backbone in float16.")


## 2. Install the Lev runtime into Colab's Python

The [card](https://huggingface.co/cisco-ai/lev-3b#installation) install is:

```bash
git clone https://github.com/splunk/lev.git lev
cd lev
python3 scripts/setup_kev.py
uv sync --frozen --extra cuda128-torch28   # or cuda124-torch26
source .venv/bin/activate
```

Colab already has a CUDA torch. **Do not create a uv venv** and **do not reinstall torch** — that often drops GPU support. This cell clones Lev (commit `dc80fb53a90993208c3c20172cfab4884e1a1049`), runs `scripts/setup_kev.py` (fetches [Kev](https://github.com/jaredpalmer/kev) at the pinned revision and applies Lev's patch), and `pip install -e`s `third_party/kev` **without its torch pin**. Inference deps are installed separately: `transformers==5.20.1` (same pin as the Granite 4.1 T4 notebook in this repo), `peft>=0.21` (the adapter was saved with PEFT 0.21.0), `accelerate`, `pydantic`, `huggingface_hub`, and `datasets` (imported when you `from kev.evaluate import load`).

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch. Skip the CUDA 12.8 extra (`flash-linear-attention`, `causal-conv1d`, `triton==3.8.0`). Do **not** install `flash-attn`.

Colab's preinstalled **`torchao==0.10.0`** makes `peft>=0.21` raise `ImportError` (`only versions above 0.16.0 are supported`). This notebook does not use torchao, so the install cell **uninstalls it** rather than upgrading it on top of Colab's torch. Keep Colab's torch; kev's `numpy>=2.5.3` / `scikit-learn>=1.9.1` / `torch<2.9` pins are skipped (`--no-deps`).

An editable `pip install -e` is often **not** on `sys.path` until the session restarts. This cell inserts `third_party/kev` onto `sys.path` and invalidates import caches so **Runtime > Run all** can `import kev` without a restart. The install cell then drops any already-imported `transformers` / `peft` / `torchao` modules so 5.20.1 is visible. If the version assert still fails, restart once and continue from the GPU cell.


In [ ]:
import importlib
import importlib.metadata as md
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

LEV_COMMIT = "dc80fb53a90993208c3c20172cfab4884e1a1049"
LEV_DIR = Path("/content/lev") if Path("/content").is_dir() else Path("lev").resolve()

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

if not (LEV_DIR / ".git").is_dir():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/splunk/lev.git", str(LEV_DIR)]
    )
head = subprocess.check_output(["git", "-C", str(LEV_DIR), "rev-parse", "HEAD"], text=True).strip()
if head != LEV_COMMIT:
    subprocess.check_call(["git", "-C", str(LEV_DIR), "fetch", "--depth", "1", "origin", LEV_COMMIT])
    subprocess.check_call(["git", "-C", str(LEV_DIR), "checkout", "--detach", LEV_COMMIT])
print("lev commit", subprocess.check_output(["git", "-C", str(LEV_DIR), "rev-parse", "HEAD"], text=True).strip())

kev_src = LEV_DIR / "third_party" / "kev"
if not (kev_src / "kev").is_dir():
    subprocess.check_call([sys.executable, str(LEV_DIR / "scripts" / "setup_kev.py")])
print("kev source", kev_src)

# Card uses a uv extra that pins torch. Keep Colab's CUDA wheel; install kev without deps.
get_ipython().run_line_magic("pip", f"install -q -e {kev_src} --no-deps")
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers==5.20.1" "peft>=0.21" "accelerate>=1.15.0" '
    '"pydantic>=2.9" "huggingface_hub>=0.26" datasets safetensors',
)
# peft 0.21.x raises ImportError on Colab's torchao 0.10.0 (wants >=0.16).
# We do not use torchao; uninstall rather than upgrading on top of Colab's torch.
get_ipython().run_line_magic("pip", "uninstall -y torchao")

# Editable -e is often invisible until restart. Put the source on sys.path now.
if str(kev_src) not in sys.path:
    sys.path.insert(0, str(kev_src))
importlib.invalidate_caches()

# Drop pre-imported Colab copies so 5.20.1 / peft see no torchao 0.10.0.
for name in list(sys.modules):
    if (
        name == "transformers"
        or name.startswith("transformers.")
        or name == "peft"
        or name.startswith("peft.")
        or name == "torchao"
        or name.startswith("torchao.")
        or name == "kev"
        or name.startswith("kev.")
    ):
        del sys.modules[name]

import transformers
from packaging.version import Version

assert Version(transformers.__version__.split("+")[0]) >= Version("5.17"), (
    f"transformers {transformers.__version__} is loaded; Lev/Kev needs >=5.17. "
    "Runtime > Restart session, then rerun from the GPU check cell."
)
import kev  # noqa: F401 — prove the path works without a session restart

print("transformers", transformers.__version__)
print("peft", md.version("peft"))
try:
    print("torchao", md.version("torchao"), "(should be gone)")
except md.PackageNotFoundError:
    print("torchao: not installed (good)")
print("kev", kev.__file__)
print("install done — skipped flash-attn, flash-linear-attention, causal-conv1d, uv venv")


> If the version assert failed, use *Runtime > Restart session* and continue from the GPU-check cell (the pin is already installed). The git clone on disk survives.


## 3. Load Lev-3B (pointer head, not PEFT-only)

`hf download cisco-ai/lev-3b` as in the card, then `load(str(model_dir), dev="cuda", dtype=...)`. The loader restores adapters **and** `head.pt`. It also fetches the pinned Granite base.

Two T4-specific details:

1. **`load()` forces `dtype=torch.bfloat16` when `head.pt` has `weights_dtype=bf16`.** That matches the card on Ampere+. A T4 has no native bf16, so this notebook temporarily clears that flag and passes **`dtype=torch.float16`**. LoRA / structural-token rows / the pointer head are still restored in float32 (`restore_adapter_dtypes`).
2. **Attention is SDPA, then eager.** Do not request FlashAttention 2.

`temperature.json` is **not** applied automatically. The load cell sets `model.head.temperature = 1.0` (raw probabilities). Demo 1 also evaluates the card's fitted scalar.


In [ ]:
import gc
import importlib
import importlib.metadata as md
import json
import sys
from pathlib import Path

# Same clone as the install cell. Editable -e is often missing from sys.path until restart.
LEV_DIR = Path("/content/lev") if Path("/content").is_dir() else Path("lev").resolve()
kev_src = LEV_DIR / "third_party" / "kev"
if kev_src.is_dir() and str(kev_src) not in sys.path:
    sys.path.insert(0, str(kev_src))
importlib.invalidate_caches()

# peft 0.21.x raises if Colab's torchao 0.10.0 is still installed (wants >=0.16).
try:
    md.version("torchao")
except md.PackageNotFoundError:
    pass
else:
    get_ipython().run_line_magic("pip", "uninstall -y torchao")
for name in list(sys.modules):
    if (
        name == "torchao"
        or name.startswith("torchao.")
        or name == "peft"
        or name.startswith("peft.")
    ):
        del sys.modules[name]

from huggingface_hub import snapshot_download
from kev.api import SystemOneRequest, to_record
from kev.evaluate import load

MODEL_ID = "cisco-ai/lev-3b"
BASE_ID = "ibm-granite/granite-4.1-3b-base"
BASE_REVISION = "dacb9cb9157bec98e99b09f285c92a4d58405c96"


def print_vram(label=""):
    suffix = f" ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM{suffix}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"peak {peak:.2f} GB | free {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


def load_lev(model_dir, dev, dtype, attn):
    # Card: load(str(model_dir), dev="cuda", dtype=torch.bfloat16).
    # On a T4, honour `dtype` even when head.pt records weights_dtype=bf16.
    real_torch_load = torch.load

    def patched_torch_load(*args, **kwargs):
        # PyTorch 2.6+ defaults weights_only=True; head.pt is a metadata pickle.
        kwargs.setdefault("weights_only", False)
        try:
            obj = real_torch_load(*args, **kwargs)
        except TypeError:
            kwargs.pop("weights_only", None)
            obj = real_torch_load(*args, **kwargs)
        if (
            isinstance(obj, dict)
            and obj.get("weights_dtype") == "bf16"
            and dtype != torch.bfloat16
        ):
            obj = dict(obj)
            obj["weights_dtype"] = None
        return obj

    torch.load = patched_torch_load
    try:
        return load(str(model_dir), dev=dev, dtype=dtype, attn=attn)
    finally:
        torch.load = real_torch_load


MODEL_DIR = Path(snapshot_download(MODEL_ID))
print("model dir", MODEL_DIR)
print("adapter / head files:", sorted(p.name for p in MODEL_DIR.iterdir() if p.is_file()))

try:
    head_meta = torch.load(MODEL_DIR / "head.pt", map_location="cpu", weights_only=False)
except TypeError:
    head_meta = torch.load(MODEL_DIR / "head.pt", map_location="cpu")
print("head.pt base:", head_meta.get("base"), "revision:", head_meta.get("base_revision"))
print(
    "weights_dtype:",
    head_meta.get("weights_dtype"),
    "| adapter_dtype:",
    head_meta.get("adapter_dtype"),
    "| execution_mode:",
    head_meta.get("execution_mode"),
)
if head_meta.get("base") != BASE_ID or head_meta.get("base_revision") != BASE_REVISION:
    print("note: head.pt base/revision differ from the card table; using head.pt as the loader does.")

FITTED_T = json.loads((MODEL_DIR / "temperature.json").read_text())["temperature"]
print("fitted temperature (not applied yet):", FITTED_T)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tok = model = None
ATTN = last_error = None
for attn in ("sdpa", "eager"):
    try:
        tok, model = load_lev(MODEL_DIR, "cuda", DTYPE, attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} load failed:", exc)
        try:
            del model
        except Exception:
            pass
        model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError(f"Could not load {MODEL_ID} through kev.evaluate.load") from last_error

model.head.temperature = 1.0  # raw probabilities, as in the card snippet
tokenizer = tok

backbone_dtype = next(p.dtype for n, p in model.lm.named_parameters() if "lora_" not in n)
lora_dtypes = {p.dtype for n, p in model.lm.named_parameters() if "lora_" in n}
print("attention:", ATTN)
print("backbone dtype:", backbone_dtype, "| LoRA dtypes:", lora_dtypes, "| head dtype:", next(model.head.parameters()).dtype)
print("head.temperature:", model.head.temperature)
print_vram("after load")


Encode a request the way the card does: `SystemOneRequest` → `to_record` → `model.encode` → `model.probs`. Print the **full probability vector** and the selected answer. For `noul`, the card: the second probability is P(yes); binary decisions use the maximum-probability option (a 0.5 threshold; exact ties select normal / `false`).


In [ ]:
def decide(request, temperature=1.0):
    """Return (meta, probability tensors) for one System One request."""
    model.head.temperature = float(temperature)
    record, meta = to_record(SystemOneRequest.model_validate(request))
    encoded = model.encode(tokenizer, record, strict=True)
    vectors = model.probs(encoded)
    return meta, vectors


def show(meta, vectors, title=None):
    if title:
        print(f"=== {title}  (T={model.head.temperature}) ===")
    for item, tensor in zip(meta, vectors):
        p = tensor.tolist()
        qid, qtype = item["id"], item["type"]
        finite = torch.isfinite(tensor).all().item()
        if not finite:
            print(f"{qid} ({qtype}): non-finite probabilities {p}")
            print("float16 may have overflowed; reload with DTYPE = torch.float32 (~12 GB, last resort).")
            continue
        if qtype == "noul":
            labels = ["false", "true"]
            # Card: max-prob option; exact ties select normal (false).
            selected = "true" if p[1] > p[0] else "false"
            dist = dict(zip(labels, p))
            print(f"{qid} (noul): selected={selected}  P(yes)={p[1]:.6f}")
            print(f"  probabilities: {dist}")
        elif qtype == "choice":
            keys = item["keys"]
            idx = max(range(len(p)), key=lambda i: (p[i], -i))
            dist = dict(zip(keys, p))
            print(f"{qid} (choice): selected={keys[idx]!r}  p={p[idx]:.6f}")
            print(f"  probabilities: {dist}")
        else:
            legend = item["legend"]
            idx = max(range(len(p)), key=lambda i: (p[i], -i))
            expected = sum(i * pi for i, pi in enumerate(p))
            dist = {legend[str(i)]: pi for i, pi in enumerate(p)}
            print(f"{qid} (score): argmax_level={idx}  expected_level={expected:.4f}")
            print(f"  selected: {legend[str(idx)]!r}")
            print(f"  probabilities: {dist}")
    print()


## 4. Demo 1 — `noul`: is this log window anomalous?

The [card's local-inference example](https://huggingface.co/cisco-ai/lev-3b#local-inference): a two-line session, a binary question, raw `T=1`, then the same request at the fitted temperature from `temperature.json`. A second window is routine successful operation, so you can see both sides. State serialization, question wording, and candidate descriptions are part of the input and can affect results.


In [ ]:
CARD_ANOMALY = {
    "state": json.dumps(
        [
            "INFO service started",
            "ERROR request failed after repeated retries",
        ]
    ),
    "questions": {
        "is_anomalous": {
            "type": "noul",
            "instructions": "Is this log session anomalous?",
            "criteria": {
                "false": "Routine or successful normal system operation",
                "true": "Failure, error, or behavior inconsistent with normal operation",
            },
        },
    },
}

ROUTINE = {
    "state": json.dumps(
        [
            "INFO service started",
            "INFO request completed successfully in 12ms",
        ]
    ),
    "questions": CARD_ANOMALY["questions"],
}

meta, vectors = decide(CARD_ANOMALY, temperature=1.0)
show(meta, vectors, "card example, raw T=1")

meta_t, vectors_t = decide(CARD_ANOMALY, temperature=FITTED_T)
show(meta_t, vectors_t, f"card example, fitted T={FITTED_T}")

meta_ok, vectors_ok = decide(ROUTINE, temperature=1.0)
show(meta_ok, vectors_ok, "routine window, raw T=1")

print_vram("demo 1")


## 5. Demo 2 — `choice`: ATT&CK parent technique from a handful of alternatives

The card's ATT&CK eval offered **130** parent techniques. Questions and alternatives are caller-defined; this demo passes **five** MITRE ATT&CK parent techniques. The probabilities are **conditional on this closed set** — they are not P(the session is malicious). The snippet is a short encoded-PowerShell host-application log line, not a paper replication input.


In [ ]:
ATTACK = {
    "state": (
        "Nov 23 03:14:08 workstation-04 Microsoft-Windows-PowerShell: "
        "HostApplication=powershell.exe -NoP -NonI -W Hidden -EncodedCommand "
        "JABjAGwAaQBlAG4AdAAgAD0AIABOAGUAdwAtAE8AYgBqAGUAYwB0ACAAUwB5AHMAdABlAG0ALgBOAGUAdAAuAFMAbwBjAGsAZQB0AHMALgBUAEMAUABDAGwAaQBlAG4AdAA="
    ),
    "questions": {
        "parent_technique": {
            "type": "choice",
            "instructions": (
                "Which MITRE ATT&CK parent technique best matches this log snippet? "
                "Choose only among the supplied alternatives."
            ),
            "criteria": {
                "T1059": "Command and Scripting Interpreter — adversary executes via shells or scripts such as PowerShell, cmd, or bash",
                "T1003": "OS Credential Dumping — adversary dumps account credentials from the OS or a domain controller",
                "T1048": "Exfiltration Over Alternative Protocol — adversary steals data over a protocol not commonly used for exfil",
                "T1078": "Valid Accounts — adversary uses legitimate credentials to access systems",
                "T1053": "Scheduled Task/Job — adversary schedules a task or job to execute later",
            },
        },
    },
}

meta, vectors = decide(ATTACK, temperature=1.0)
show(meta, vectors, "ATT&CK parent technique, raw T=1")
print_vram("demo 2")


## 6. Demo 3 — `score`: severity 0–3

A `score` question with four ordered descriptions (lowest first). The card: use the probabilities to calculate the **expected zero-based level**. This cell reuses the card's two-line error window as the state. Print the vector, the argmax level, and the expected level.


In [ ]:
SEVERITY = {
    "state": json.dumps(
        [
            "INFO service started",
            "ERROR request failed after repeated retries",
        ]
    ),
    "questions": {
        "severity": {
            "type": "score",
            "instructions": "How severe is this log session?",
            "criteria": [
                "0 Informational: routine successful operation, no action needed",
                "1 Low: unusual but likely benign; log for later review",
                "2 Medium: suspicious failure or policy deviation; investigate",
                "3 High: likely malicious activity or a severe outage; contain or page on-call",
            ],
        },
    },
}

meta, vectors = decide(SEVERITY, temperature=1.0)
show(meta, vectors, "severity 0-3, raw T=1")
print_vram("demo 3")


## 7. Caveats and cleanup

- **Research model, not a detector you ship blindly.** The card: not a general-purpose chat model or an autonomous remediation system. Labels are imperfect, source coverage is limited, and probabilities can be unreliable under distribution shift or changes in prompts and serialization. Validate locally; keep human-reviewed action policies.
- **Not text generation.** There is no decoder output. `head.pt` is required; PEFT-only loading is incomplete.
- **HDFS / OOD.** The card: HDFS transfer remains weak; the model can miss anomalies with high confidence. The fitted temperature (`T=1.8234449771164334`) does not remove that failure mode and does not change the argmax.
- **ATT&CK scores are closed-set.** Conditional on the techniques you offered, not P(malicious).
- **Float16 on a T4.** Saved inference precision is BF16 backbone + FP32 adapters / structural-token rows / pointer head. T4 has no native bf16, so this notebook loads the backbone in float16 via a small `load()` workaround. Newer GPUs (`sm_80+`) can pass `dtype=torch.bfloat16` without clearing `weights_dtype`. No FlashAttention 2. No uv venv.
- **Context.** Saved state/branch/packed limits are `32768/65536/65536` tokens — execution limits, not a promise they fit a 15 GB T4. Demos stay short.
- **Replication.** The card: use the paper's prepared inputs for replication. These demos illustrate the interface.
- **Apache-2.0** for the Lev adapter and pointer head. Granite base and training-data sources keep their own terms; raw training data are not included.

The next cell drops the model so you can load something else on the same T4.


In [ ]:
print_vram("before cleanup")
for name in ("model", "tokenizer", "tok", "head_meta"):
    try:
        del globals()[name]
    except KeyError:
        pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Lev-3B released. Re-run the load cell if you want another request.")
